In [10]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="Voxel51/gaussian_splatting",
    filename="FO_dataset/truck/point_cloud/iteration_30000/point_cloud.ply",
    repo_type="dataset",
    local_dir="data",
)
print(f"downloaded: {path}")

downloaded: data/FO_dataset/truck/point_cloud/iteration_30000/point_cloud.ply


In [1]:
from plyfile import PlyData
import numpy as np

ply = PlyData.read('data/FO_dataset/truck/point_cloud/iteration_30000/point_cloud.ply')
v = ply['vertex']
print(f'num gaussians: {len(v)}')
print(f'properties: {[p.name for p in v.properties]}')

num gaussians: 2056645
properties: ['x', 'y', 'z', 'nx', 'ny', 'nz', 'f_dc_0', 'f_dc_1', 'f_dc_2', 'f_rest_0', 'f_rest_1', 'f_rest_2', 'f_rest_3', 'f_rest_4', 'f_rest_5', 'f_rest_6', 'f_rest_7', 'f_rest_8', 'f_rest_9', 'f_rest_10', 'f_rest_11', 'f_rest_12', 'f_rest_13', 'f_rest_14', 'f_rest_15', 'f_rest_16', 'f_rest_17', 'f_rest_18', 'f_rest_19', 'f_rest_20', 'f_rest_21', 'f_rest_22', 'f_rest_23', 'f_rest_24', 'f_rest_25', 'f_rest_26', 'f_rest_27', 'f_rest_28', 'f_rest_29', 'f_rest_30', 'f_rest_31', 'f_rest_32', 'f_rest_33', 'f_rest_34', 'f_rest_35', 'f_rest_36', 'f_rest_37', 'f_rest_38', 'f_rest_39', 'f_rest_40', 'f_rest_41', 'f_rest_42', 'f_rest_43', 'f_rest_44', 'opacity', 'scale_0', 'scale_1', 'scale_2', 'rot_0', 'rot_1', 'rot_2', 'rot_3']


In [2]:
# inspect one gaussian
v[0]

np.void((-0.1597827, 1.4494442, -0.6166176, 0.0, 0.0, 0.0, 0.46016848, 0.33630508, 0.59090793, -0.071440466, -0.0013364835, -0.051153146, 0.02389667, -0.040006734, -0.10956806, -0.042897493, 0.001680109, -0.11900488, -0.017594125, 0.05447582, -0.039647236, -0.04082001, -0.03610718, 0.05891986, -0.07523063, 0.008073062, -0.0382598, 0.02781504, -0.042926673, -0.103530385, -0.045755595, -0.012599778, -0.109099194, -0.0044407267, 0.05249185, -0.06958392, -0.034626603, -0.013631118, 0.07888801, -0.097184815, -0.0028376516, -0.046891846, 0.028167224, -0.050467562, -0.11080455, -0.034952097, -0.040618412, -0.10923483, -0.025447905, 0.049914718, -0.08089487, -0.029292906, -0.052923217, 0.06224407, -0.50609946, -4.8696566, -11.619647, -5.8555455, 0.8774532, 0.03675382, -0.32422936, 0.037720993), dtype=[('x', '<f4'), ('y', '<f4'), ('z', '<f4'), ('nx', '<f4'), ('ny', '<f4'), ('nz', '<f4'), ('f_dc_0', '<f4'), ('f_dc_1', '<f4'), ('f_dc_2', '<f4'), ('f_rest_0', '<f4'), ('f_rest_1', '<f4'), ('f_rest_

In [2]:
xyz = np.stack([v['x'], -v['y'], v['z']], axis=1)

# DC SH term: SH_C0 * f_dc + 0.5, then sigmoid to get RGB in [0,1]
SH_C0 = 0.28209479177387814
f_dc = np.stack([v['f_dc_0'], v['f_dc_1'], v['f_dc_2']], axis=1)
rgb = np.clip(SH_C0 * f_dc + 0.5, 0, 1)

print(f'xyz range: {xyz.min(axis=0)} to {xyz.max(axis=0)}')
print(f'rgb range: {rgb.min(axis=0)} to {rgb.max(axis=0)}')

xyz range: [-81.489105 -26.058327 -98.11391 ] to [44.182583 28.49416  58.058346]
rgb range: [0. 0. 0.] to [1. 1. 1.]


In [6]:
import open3d as o3d

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz)
pcd.colors = o3d.utility.Vector3dVector(rgb)

o3d.visualization.draw_geometries([pcd], window_name="truck", width=1280, height=720)

In [3]:
def build_cov(scale, quat):
    s = np.exp(scale)
    w, x, y, z = quat[:,0], quat[:,1], quat[:,2], quat[:,3]

    R = np.stack([
        1-2*(y*y+z*z),  2*(x*y-w*z),   2*(x*z+w*y),
        2*(x*y+w*z),    1-2*(x*x+z*z), 2*(y*z-w*x),
        2*(x*z-w*y),    2*(y*z+w*x),   1-2*(x*x+y*y)
    ], axis=1).reshape(-1, 3, 3)

    S2 = np.zeros((len(scale), 3, 3))
    S2[:,0,0] = s[:,0]**2
    S2[:,1,1] = s[:,1]**2
    S2[:,2,2] = s[:,2]**2

    cov = R @ S2 @ R.transpose(0, 2, 1)
    return np.stack([
        cov[:,0,0], cov[:,0,1], cov[:,0,2],
        cov[:,1,1], cov[:,1,2], cov[:,2,2]
    ], axis=1)

scale = np.stack([v['scale_0'], v['scale_1'], v['scale_2']], axis=1).astype(np.float32)
quat  = np.stack([v['rot_0'],   v['rot_1'],   v['rot_2'],   v['rot_3']], axis=1).astype(np.float32)
quat  = quat / np.linalg.norm(quat, axis=1, keepdims=True)
cov   = build_cov(scale, quat)

opacity = (1 / (1 + np.exp(-v['opacity'].astype(np.float32))))

print(f'cov sample:\n{cov[:3]}')
print(f'opacity range: {opacity.min():.3f} to {opacity.max():.3f}')
print(f'scale range: {np.exp(scale).min():.4f} to {np.exp(scale).max():.4f}')

cov sample:
[[ 3.71992153e-05  2.68940099e-06  2.50542326e-05  2.21546086e-07
   1.22252115e-06  2.97026635e-05]
 [ 1.39180136e-04  1.38658595e-04 -1.52587160e-04  5.75825093e-04
  -1.05124633e-04  1.72439864e-04]
 [ 1.97057156e-05 -1.10058836e-05 -1.04213562e-05  8.88686248e-05
   2.54575769e-05  1.33150322e-05]]
opacity range: 0.002 to 1.000
scale range: 0.0000 to 29.2560


In [31]:
import os
os.makedirs('public', exist_ok=True)

OPACITY_THRESHOLD = 0.1
SIZE_THRESHOLD    = 0.01  # world-space units; max extent across all 3 axes

# max extent of each Gaussian: largest of the 3 exp(scale) values
max_scale = np.exp(scale).max(axis=1)

mask = (opacity > OPACITY_THRESHOLD) & (max_scale > SIZE_THRESHOLD)
print(f'kept {mask.sum()} / {len(v)} gaussians ({100*mask.mean():.1f}%)')

xyz_out     = xyz[mask].astype(np.float32)
opacity_out = opacity[mask].astype(np.float32)
f_dc_raw    = np.stack([v['f_dc_0'], v['f_dc_1'], v['f_dc_2']], axis=1).astype(np.float32)[mask]
f_rest_R    = np.stack([v[f'f_rest_{i}'] for i in range( 0, 15)], axis=1).astype(np.float32)[mask]
f_rest_G    = np.stack([v[f'f_rest_{i}'] for i in range(15, 30)], axis=1).astype(np.float32)[mask]
f_rest_B    = np.stack([v[f'f_rest_{i}'] for i in range(30, 45)], axis=1).astype(np.float32)[mask]
cov_out     = cov[mask].astype(np.float32)
pad         = np.zeros((mask.sum(), 6), dtype=np.float32)

data = np.concatenate([
    xyz_out,                          # 0-2
    f_dc_raw,                         # 3-5
    f_rest_R,                         # 6-20
    f_rest_G,                         # 21-35
    f_rest_B,                         # 36-50
    opacity_out.reshape(-1, 1),       # 51
    cov_out,                          # 52-57
    pad,                              # 58-63
], axis=1)
assert data.shape[1] == 64

data.tofile('public/truck.bin')
print(f'written: public/truck.bin  ({data.nbytes / 1e6:.1f} MB)  —  {mask.sum()} gaussians x 256 bytes')

kept 873575 / 2056645 gaussians (42.5%)
written: public/truck.bin  (223.6 MB)  —  873575 gaussians x 256 bytes
